# 91. Decode Ways
**Difficulty:** 🟡 Medium · **Topic:** Dynamic Programming · **LeetCode:** https://leetcode.com/problems/decode-ways/

## 💡 Concepts

**Core concept(s):** 1-D DP over positions — ways to decode up to `i` = ways using a 1-digit letter + ways using a valid 2-digit letter.

**Why it applies here:** Letters map A–Z to 1–26. At each position you may take one digit (if not '0') or two digits (if they form 10–26). So the count at `i` combines the counts one and two positions back — a Fibonacci-like recurrence with validity checks.

**Key intuition:** Ways up to here = (ways before, if this digit is 1–9) + (ways two back, if these two digits are 10–26).

---

### 📚 What is Dynamic Programming (DP)?
**DP** solves a big problem by solving smaller **overlapping** subproblems once and reusing the answers. Two styles: **memoization** (recursion that caches results) and **tabulation** (fill a table from the smallest cases up).
- **Why it's fast:** it turns exponential re-computation into a single sweep over the subproblems.
- **In Python:** a `dict`/list cache, or a `dp` list/2-D table.

### 📚 Subproblems & Recurrence
The heart of DP is a **recurrence**: the answer for a state written in terms of smaller states (e.g. `dp[i] = dp[i-1] + dp[i-2]`). Find the recurrence and the base cases, and the code writes itself.

---

**Prerequisite knowledge:**
- The 1-digit / 2-digit split.
- Careful handling of '0'.

## 📝 Problem

A message of digits maps to letters (`1→A ... 26→Z`). How many ways can it be decoded? Leading zeros are invalid.

**Example**
```
"12"  -> 2   ("AB", "L")
"226" -> 3   ("BZ","VF","BBF")
"06"  -> 0
```

> Two approaches: exponential brute force and `O(n)` DP (`O(1)` space).

### Approach 1 — Brute Recursion (worst)

**Idea:** At each position, take one digit (if valid) and/or two digits (if 10–26), summing the ways.

**Time:** `O(2^n)`. **Space:** `O(n)`.

In [ ]:
def decode_brute(s):
    def dfs(i):                            # number of ways to decode s[i:]
        if i == len(s):
            return 1                       # reached the end -> one valid decoding
        if s[i] == "0":
            return 0                       # no letter starts with 0 -> dead end
        res = dfs(i + 1)                   # take this digit alone (1-9)
        if i + 1 < len(s) and int(s[i:i+2]) <= 26:
            res += dfs(i + 2)              # take two digits together (10-26)
        return res
    return dfs(0)

### Approach 2 — Rolling DP (optimal)

**Idea:** Track ways up to the previous two positions. At each step add the 1-digit option (if the digit isn't 0) and the 2-digit option (if 10–26).

**Time:** `O(n)`. **Space:** `O(1)`.

In [ ]:
def decode_dp(s):
    if not s or s[0] == "0":
        return 0                           # a leading 0 can't be decoded
    prev2, prev1 = 1, 1                     # ways to decode up to positions i-2 and i-1
    for i in range(1, len(s)):
        cur = 0
        if s[i] != "0":
            cur += prev1                   # this digit stands alone (1-9)
        if 10 <= int(s[i-1:i+1]) <= 26:
            cur += prev2                   # this digit pairs with the previous (10-26)
        prev2, prev1 = prev1, cur          # slide the window forward
    return prev1

In [ ]:
# Correctness check
tests = [("12",2),("226",3),("06",0),("0",0),("10",1),("11106",2),("2101",1)]
for s, exp in tests:
    a, b = decode_brute(s), decode_dp(s)
    print(f"{s!r} -> brute={a}, dp={b} | expected={exp}")
    assert a == b == exp, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on inputs of growing size `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(n)`        | ≈ **2×** |
| `O(n log n)`  | ≈ **2×** (slightly more) |
| `O(n²)`       | ≈ **4×** |

Inputs are shaped to force the worst case. (Exponential brute-force versions are shown in the code but omitted from timing where they would blow up — noted per notebook.)

*(Brute force omitted; the DP is timed on a bounded-count input so integer size doesn't skew results.)*

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    s = ('10' * (n // 2 + 1))[:n]   # each '10' decodes exactly one way -> counts stay small
    return (s,)
solutions = {
    "dp O(n) space O(1)": decode_dp,
}
sizes = [20000, 40000, 80000, 160000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Fibonacci-style DP with validity checks:** combine the previous one/two states, gated by rules.
- **Zeros are the trap:** a leading '0' kills a decoding; only 10–26 form a valid pair.
- **Signal:** "count the ways to parse/segment a sequence with 1- or 2-unit steps".
- **Related problems:** Climbing Stairs, Decode Ways II, Fibonacci.
- **Common pitfalls:** (1) mishandling '0'; (2) allowing pairs like 27–99 or 00–09.